# Upload vidgrep output files to S3

Finds scan/inventory/worker output files (`.jsonl`, `.json`, `.csv`, and optionally clip `.mp4`s),
then uploads them to an S3 bucket with boto3.

**Credentials:** uses the standard AWS credential chain — environment variables
(`AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY`), `~/.aws/credentials`, or an SSO/assumed-role
profile set via `AWS_PROFILE` below.

Workflow: edit the **Configuration** cell, run everything top to bottom. `DRY_RUN = True`
previews what would be uploaded without touching S3.

In [ ]:
# One-time setup (safe to re-run)
%pip install --quiet boto3

## Configuration

In [ ]:
from pathlib import Path

# --- S3 destination ---
BUCKET = "my-bucket-name"          # <-- change me
PREFIX = "vidgrep/"                # key prefix inside the bucket ("" for bucket root)
AWS_PROFILE = None                  # e.g. "default" or an SSO profile; None = default chain
AWS_REGION = None                   # e.g. "us-west-2"; None = profile/env default

# --- What to upload ---
SOURCE_DIRS = [Path("../data")]    # directories to search (e.g. [Path(r"D:\\scans"), Path("../data")])
RECURSIVE = False                   # search subdirectories too
PATTERNS = ["*.jsonl", "*.json", "*.csv"]  # add "*_clip*.mp4" or "*.mp4" to include clips
EXCLUDE_NAMES = {".agent.state.json", "package.json", "pyproject.toml"}

# --- Behaviour ---
DRY_RUN = True                      # True = list what would upload, upload nothing
SKIP_IF_SAME_SIZE = True            # skip files already in S3 with an identical byte size
KEEP_DIR_STRUCTURE = False          # True = key mirrors path relative to its source dir;
                                    # False = flat keys (PREFIX + filename)

## Discover output files

In [ ]:
def human_size(n: int) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB":
            return f"{n:.1f} {unit}" if unit != "B" else f"{n} B"
        n /= 1024


def discover_files():
    found = {}
    for src in SOURCE_DIRS:
        src = src.resolve()
        if not src.is_dir():
            print(f"WARNING: not a directory, skipping: {src}")
            continue
        for pattern in PATTERNS:
            matches = src.rglob(pattern) if RECURSIVE else src.glob(pattern)
            for path in matches:
                if path.is_file() and path.name not in EXCLUDE_NAMES:
                    found[path] = src
    return found


files = discover_files()
total = sum(p.stat().st_size for p in files)
print(f"Found {len(files)} file(s), {human_size(total)} total:\n")
for path in sorted(files):
    print(f"  {human_size(path.stat().st_size):>10}  {path}")

## Connect to S3 and verify credentials

In [ ]:
import boto3
from botocore.exceptions import ClientError

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
s3 = session.client("s3")

identity = session.client("sts").get_caller_identity()
print(f"Authenticated as: {identity['Arn']}")

try:
    s3.head_bucket(Bucket=BUCKET)
    print(f"Bucket OK: s3://{BUCKET}")
except ClientError as exc:
    code = exc.response["Error"]["Code"]
    raise SystemExit(
        f"Cannot access bucket '{BUCKET}' ({code}). "
        "Check the bucket name, region, and your permissions."
    )

## Upload

In [ ]:
import sys


def s3_key_for(path: Path, source_dir: Path) -> str:
    if KEEP_DIR_STRUCTURE:
        rel = path.relative_to(source_dir).as_posix()
    else:
        rel = path.name
    return f"{PREFIX}{rel}"


def remote_size(key: str):
    try:
        return s3.head_object(Bucket=BUCKET, Key=key)["ContentLength"]
    except ClientError as exc:
        if exc.response["Error"]["Code"] in ("404", "NoSuchKey", "NotFound"):
            return None
        raise


class Progress:
    def __init__(self, path: Path):
        self.name = path.name
        self.total = path.stat().st_size
        self.seen = 0

    def __call__(self, bytes_transferred):
        self.seen += bytes_transferred
        pct = 100 * self.seen / self.total if self.total else 100
        sys.stdout.write(f"\r    {self.name}: {pct:5.1f}%")
        sys.stdout.flush()


uploaded, skipped, failed = [], [], []

for path in sorted(files):
    key = s3_key_for(path, files[path])
    size = path.stat().st_size

    if SKIP_IF_SAME_SIZE and not DRY_RUN and remote_size(key) == size:
        skipped.append(key)
        print(f"  skip (already uploaded)  s3://{BUCKET}/{key}")
        continue

    if DRY_RUN:
        print(f"  [dry run] would upload {path}  ->  s3://{BUCKET}/{key}  ({human_size(size)})")
        continue

    try:
        s3.upload_file(str(path), BUCKET, key, Callback=Progress(path))
        print(f"  ->  s3://{BUCKET}/{key}")
        uploaded.append(key)
    except Exception as exc:
        print(f"\n  FAILED {path}: {exc}")
        failed.append((path, exc))

print()
if DRY_RUN:
    print(f"Dry run complete — {len(files)} file(s) would be uploaded. "
          "Set DRY_RUN = False and re-run to upload.")
else:
    print(f"Uploaded {len(uploaded)}, skipped {len(skipped)}, failed {len(failed)}.")
    if failed:
        print("Failed files:")
        for path, exc in failed:
            print(f"  {path}: {exc}")

## Verify: list what's now under the prefix

In [ ]:
paginator = s3.get_paginator("list_objects_v2")
count, total = 0, 0
for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX):
    for obj in page.get("Contents", []):
        count += 1
        total += obj["Size"]
        print(f"  {human_size(obj['Size']):>10}  s3://{BUCKET}/{obj['Key']}")
print(f"\n{count} object(s), {human_size(total)} under s3://{BUCKET}/{PREFIX}")